In [1]:
import sys
import os
# Add the parent directory to sys.path so we can import dyntex
sys.path.append(os.path.abspath('..'))

import numpy as np
import imageio
import torch as tch
from dyntex.MotionCloud import MotionCloud
import config


In [2]:
def generate_motion_cloud(
    th=config.LOOP_CONFIG['th_start'],
    out_path='motioncloud.mp4',
    duration_sec=config.LOOP_CONFIG['duration_sec'],
    N=config.MC_CONFIG['N'],
    fps=config.MC_CONFIG['fps'],
    contrast=config.MC_CONFIG['contrast'],
    ave_lum=config.MC_CONFIG['ave_lum'],
    over_samp=config.MC_CONFIG['over_samp'],
    kernel_type=config.MC_CONFIG['kernel_type'],
    sf=config.MC_CONFIG['sf'],
    sf_sig=config.MC_CONFIG['sf_sig'],
    th_sig=config.MC_CONFIG['th_sig'],
    tf=config.MC_CONFIG['tf'],
    spd_scalar=config.MC_CONFIG['spd_scalar'],
    spd_dir=config.MC_CONFIG['spd_dir'],
    octa=config.MC_CONFIG['octa']
):
    """
    Generate a single Motion Cloud with the specified parameters.
    All parameters default to those defined in config.py.
    """
    # 1) Initialize MotionCloud
    MC = MotionCloud(
        N=N,
        frame_per_second=fps,
        contrast=contrast,
        ave_lum=ave_lum,
        over_samp=over_samp,
        verbose=0
    )
    
    # 2) Set parameters
    MC.set_all(
        kernel_type=kernel_type,
        sf=sf,
        sf_sig=sf_sig,
        th=th,
        th_sig=th_sig,
        tf=tf,
        spd_scalar=spd_scalar,
        spd_dir=spd_dir,
        octa=octa
    )
    
    MC.burnout() # warm-up the AR recursion
    
    # 3) Setup video writing
    n_frames = int(fps * duration_sec)
    writer = imageio.get_writer(
        out_path,
        fps=fps,
        codec='libx264',
        quality=8
    )
    
    # 4) Generate frames & write
    for i in range(n_frames):
        MC.set_fourier_translation()
        im = MC.get_frame(adjust=True)
        arr = im.detach().cpu().numpy()
        
        # Convert to unit8
        if arr.max() > 255 or arr.min() < 0:
            arr = np.clip(arr, 0, 255)
        frame_uint8 = arr.astype(np.uint8)
        writer.append_data(frame_uint8)
        
    writer.close()
    print(f'Wrote {n_frames} frames -> {out_path}')


In [3]:
# Generate motion clouds in a loop using the config dictionary
loop_cfg = config.LOOP_CONFIG

for i in range(loop_cfg['num_clouds']):
    # Incrementally increase orientation by the step size
    th_current = loop_cfg['th_start'] + i * loop_cfg['orientation_step']
    out_file = f'motioncloud_th_{th_current:.1f}.mp4'
    
    print(f'Generating MotionCloud {i+1}/{loop_cfg["num_clouds"]} with orientation {th_current} degrees...')
    generate_motion_cloud(
        th=th_current,
        out_path=out_file
    )

print('\nAll generation tasks complete!')


Generating MotionCloud 1/4 with orientation 0.0 degrees...
Wrote 120 frames -> motioncloud_th_0.0.mp4
Generating MotionCloud 2/4 with orientation 45.0 degrees...
Wrote 120 frames -> motioncloud_th_45.0.mp4
Generating MotionCloud 3/4 with orientation 90.0 degrees...
Wrote 120 frames -> motioncloud_th_90.0.mp4
Generating MotionCloud 4/4 with orientation 135.0 degrees...
Wrote 120 frames -> motioncloud_th_135.0.mp4

All generation tasks complete!


In [ ]:
# Generate motion clouds in a loop using the config dictionary
loop_cfg = config.LOOP_CONFIG

for i in range(loop_cfg['num_clouds']):
    # Incrementally increase orientation by the step size
    th_current = loop_cfg['th_start'] + i * loop_cfg['orientation_step']
    out_file = f'motioncloud_th_{th_current:.1f}.mp4'
    
    print(f'Generating MotionCloud {i+1}/{loop_cfg["num_clouds"]} with orientation {th_current} degrees...')
    generate_motion_cloud(
        th=th_current,
        out_path=out_file
    )

print('\nAll generation tasks complete!')
